# Executive Summary
This notebook applies **Principal Component Analysis (PCA)** to the MNIST handwritten digits dataset (784 features per image). PCA reduces dimensionality while retaining most variance, enabling efficient visualization and computation.  
We evaluate how PCA impacts classification performance with **K-Nearest Neighbors (KNN)** and identify an optimal trade-off between accuracy and dimensionality.

---

# Workflow
- Important Things to Know Before Applying PCA
- Applying PCA with Scikit-learn
- Identifying the optimum number of principal components
- Train KNN classifiers on different PCA dimensions
- Visualise the data first in 2D (with the first 2 PCA), then in 3D using the first 3 PCA

# Important Things to Know Before Applying PCA

Principal Component Analysis (PCA) is a popular dimensionality reduction technique.  
Here are the key concepts and considerations to keep in mind:


## 1. Why Use PCA?
- **Dimensionality Reduction**: Removes redundancy in features, making models faster and often more accurate.  
- **Visualization**: Reduces data to 2D or 3D for easier plotting and exploration.  
- **Noise Reduction**: By keeping only the main components, PCA can remove less informative variations.


## 2. Core Idea
PCA creates **new features (Principal Components)** that are linear combinations of the original ones.  
- The **first component (PC1)** captures the maximum variance in the data.  
- The **second component (PC2)** is orthogonal to PC1 and captures the next highest variance, and so on.  
- You can then keep the top *k* components that explain most of the variance.


## 3. Mathematical Backbone
PCA relies on two key steps:
1. **Covariance Matrix** – shows how features vary with respect to each other.  
2. **Eigenvalues & Eigenvectors** –  
   - Eigenvectors = Principal Components (directions of maximum variance).  
   - Eigenvalues = How much variance each component explains.  
   - Sorting eigenvalues tells you which components matter most.


## 4. Choosing the Number of Components
- Use the **explained variance ratio** to see how much information each component retains.  
- A common practice: choose the smallest number of components that explain ~90–95% of total variance.  
- You can use a cumulative explained variance curve to decide.


## 5. How PCA is Applied (Step-by-Step)
1. **Standardize Data** (zero mean, unit variance) – ensures all features contribute fairly.  
2. **Compute Covariance Matrix** of the data.  
3. **Find Eigenvalues & Eigenvectors** → gives principal components.  
4. **Select Top Components** (based on explained variance).  
5. **Project Data** onto these new components.

# Dataset Setup and Preparation

## Importing Libraries and Dataset
We used Scikit-learn for PCA, scaling, splitting, and KNN classification.

In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA

### Load Dataset
We use Kaggle’s **Digit Recognizer** dataset. Each row is a 28×28 pixel image flattened into 784 features, with the label as the target digit (0–9).


In [ ]:
# Load
df = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")

# Extracting X, Y features
Y = df["label"]
X = df.drop(columns=["label"])
# Quick peek at the first rows
df.head()


In [ ]:
print("Shape:", X.shape, "Labels:", Y.shape)

The dataset has 42,000 training images, each represented as a flattened vector of 784 pixel features (28×28).

**Drawing a sample from the dataset**

In [ ]:
samp = df.sample()
samp

In [ ]:
# Image of that sample
image = samp.iloc[:, 1:].values.reshape(28, 28)
plt.imshow(image)
plt.show()

## Train/Validation Split
We split into train/validation (80/20), stratified to maintain class balance.


In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

## Scaling
Before PCA, scaling can improved result:

Each pixel must be placed on the same scale (0 mean, unit variance).

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Applying PCA using Scikit-learn
How many components are needed to retain most of the dataset’s information?  

We fit PCA without limiting components, then plot the **cumulative explained variance**.  


In [ ]:
# with 784 pca
pca = PCA(n_components=None)
X_train_trf = pca.fit_transform(X_train)
X_test_trf = pca.transform(X_test)
X_train_trf.shape

Since the MNIST dataset has 784 features (28×28 pixels), PCA produce 784 principal components.  
However, to keep things simple and focus on the internal mechanics, we only examine the **first 3 principal components (PCs)**.

For each of these PCs, we will look at:
- **Eigenvalue** → how much variance that component captures.  
- **Eigenvector** → the direction (weighting of original features) that defines the component.  
- **Percentage of variance explained** → contribution of the component to the overall dataset variance.  

By studying just the top 3, we can understand how PCA compresses high-dimensional data into a smaller set of components while retaining the majority of the variance.


## PCA Variance Explained
#### Inspecting PCA Outputs (**Using Only first 3 PCA**) 


In [ ]:
pca = PCA(n_components=3)
X_train_tr3 = pca.fit_transform(X_train)
X_test_tr3 = pca.transform(X_test)

#### Once PCA is fitted, we can look inside the decomposition to understand what it found:

In [ ]:
# eigen vectors 
pca.components_

In [ ]:
# these are the eigenvalues for PCA 3  (it's not percentage)
pca.explained_variance_

In [ ]:
#percentage variance explained by each eigen values/vectors (1st pca 5.7%,2nd pca 4.15%)
pca.explained_variance_ratio_ *100

### PCA Components, Eigenvalues, Variance Explained

- pca.components_ → These are the directions (eigenvectors) that PCA finds. Each row is a principal component, and the numbers are the weights for each feature.

- pca.explained_variance_ → These are the eigenvalues, which show how much variance each component captures. PC1 ≈ 40.7, PC2 ≈ 29.2, PC3 ≈ 26.7

- pca.explained_variance_ratio_ → This tells you the percentage of total variance explained by each component. PC1 ≈ 5.8%, PC2 ≈ 4.1%, PC3 ≈ 3.8%

Top PCs capture only a small share individually, but together build up most variance — hence many components are needed (~150–200 for 95%+).


# Finding Optimum number of Principle Components  
We select the number of principal components such that the **cumulative variance explained** by the first *n* components is at least **90%**.  
This ensures that the reduced dataset retains most of the important information while significantly lowering dimensionality.


In [ ]:
# with 784 pca
pca = PCA(n_components=None)
X_train_trf = pca.fit_transform(X_train)
X_test_trf = pca.transform(X_test)

In [ ]:
# cumulative variance explained
np.cumsum(pca.explained_variance_ratio_*100)

In [ ]:
# plot cumulative variance
plt.plot(np.cumsum(pca.explained_variance_ratio_*100))

**Observation**  
- The first ~50 PCs capture a large fraction of variance.  
- Around **150–200 PCs** explain ~95% of total variance.  
- This reduces dimensionality by nearly **5×** (784 → ~150–200) with minimal loss of information.  

**Decision:**  
For our analysis, we will retain the **first 200 principal components**.

# KNN Classification with PCA
#### Since we have already selected our principal components, we will now evaluate how dimensionality affects classification accuracy.  

We try KNN with different PCA sizes: `[5, 50, 100, 150, 200, 300,784]`.  

In [ ]:
for i in (5,50,100,150,200,250,300,784):
    pca = PCA(n_components=i)
    
    X_train_trf = pca.fit_transform(X_train)
    X_test_trf = pca.transform(X_test)

    knn = KNeighborsClassifier()

    knn.fit(X_train_trf,Y_train)

    y_pred = knn.predict(X_test_trf)

    print(accuracy_score(Y_test,y_pred))

**Results** 
- Using only 5 components performs poorly (~0.73).
- Accuracy jumps when you use 50 components (~0.95).
- After ~200 components, accuracy stabilizes and slightly declines.

# PCA Visualization (2D & 3D)
Let’s see how digits look when projected into fewer dimensions:  

- **2D Plot (PC1 vs PC2)** → gives us a rough sense of clustering.  
- **3D Plot (PC1–PC3)** → adds another dimension for slightly better separation.  

Note: These are *visual aids only* — 2D/3D PCA projections are not sufficient for accurate classification.  


In [ ]:
# 2D projection
pca = PCA(n_components=2)
X_train_trf = pca.fit_transform(X_train)
X_test_trf = pca.transform(X_test)
X_train_trf

In [ ]:
import plotly.express as px
Y_train_trf= Y_train.astype(str)
fig = px.scatter(x=X_train_trf[:,0],
                 y=X_train_trf[:,1],
                 color=Y_train_trf,
                 color_discrete_sequence=px.colors.qualitative.G10
                ) 
fig.show()

In [ ]:
# 3D projection
pca = PCA(n_components=3)
X_train_trf = pca.fit_transform(X_train)
X_test_trf = pca.transform(X_test)
X_train_trf

In [ ]:
import plotly.express as px
Y_train_trf= Y_train.astype(str)
fig = px.scatter_3d(df, x=X_train_trf[:,0],y=X_train_trf[:,1], z= X_train_trf[:,2],
                 color=Y_train_trf)
fig.update_layout(margin=dict(l=20,r=20,t=20,b=20))
fig.show()

**Observation**  
- Some digits (like 0, 1, 7) form clear clusters.  
- Others (like 4 vs 9) overlap heavily due to handwriting similarity.  
- This explains why classification in low dimensions is difficult.  



# Key Takeaways
- PCA reduced MNIST dimensionality from **784 → ~150–200** while retaining >95% of variance.  
- Visualization in 2D/3D highlights digit clusters but also overlap between similar digits.  
- KNN classification reached ~97% accuracy with PCA-reduced features, close to using the full dataset.


